<a href="https://colab.research.google.com/github/saiDan77/nn-from-scratch/blob/main/DATAPREP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced AI Data Preparation & Pipeline Engineering

---

## Module 0: Data Profiling & Discovery

**Introduction:** We solve the "Unknown Data" problem. Before any cleaning or modeling, engineers must understand the statistical, structural, and semantic properties of the data to prevent downstream failures.

### Chapter: Automated Data Profiling & Exploratory Analysis

#### Strategic Suitability Framework (5 Factors)
1. **Problem Solved**: **Unknown dataset characteristics** that lead to incorrect preprocessing (e.g., hidden skew or incorrect types).
2. **When to Use**: Immediately after data ingestion as the very first engineering step.
3. **When to Avoid**: Never. Profiling is the prerequisite for all subsequent modules.
4. **Optimal Workflow**: **Data Summary → Missingness Analysis → Distribution Check → Cardinality Audit → Correlation Mapping**.
5. **Failure Modes**: **Over-reliance on automation**. Automated tools may miss semantic errors (e.g., a 'zip_code' stored as a float being averaged).

**Interpreting Results:** A successful profile identifies columns with >50% missingness for immediate dropping and flags high-skew features for non-linear transformation.

In [18]:
import pandas as pd
import numpy as np

def profile_feature_health(df: pd.DataFrame) -> pd.DataFrame:
    """
    Problem: Manual profiling is slow and prone to oversight.
    Solution: Automated health audit for cardinality, missingness, and type mismatch.
    """
    profile = pd.DataFrame({
        'dtype': df.dtypes,
        'missing_pct': df.isnull().mean() * 100,
        'unique_count': df.nunique(),
        'cardinality_ratio': df.nunique() / len(df)
    })

    # Flag features for Chapter 05 (High Cardinality) or Chapter 03 (Imputation)
    profile['action_hint'] = np.where(profile['missing_pct'] > 30, 'Impute/Drop',
                             np.where(profile['cardinality_ratio'] > 0.5, 'Hash/Target Encode', 'Standard Scale'))

    return profile

---

## Module: Data Versioning & Reproducibility

**Introduction:** We solve the "Reproducibility" problem. Every model must be traceable to the exact dataset version used during training to ensure auditability and debugging.

### Chapter: Dataset Version Control & Lineage

#### Strategic Suitability Framework (5 Factors)
1. **Problem Solved**: **Dataset Inconsistency**. Prevents the 'it worked on my machine' error when production data differs from training snapshots.
2. **When to Use**: Every production ML project requiring regulatory compliance or collaborative experiment tracking.
3. **When to Avoid**: Tiny educational prototypes where data is static.
4. **Optimal Workflow**: **Raw Data → Version Snapshot → Metadata Tagging → Experiment Linkage**.
5. **Failure Modes**: **Storage Bloat**. Storing full copies of massive datasets for every minor change instead of using delta-logging.

**Interpreting Results:** Success is achieved when any model ID can be mapped back to a specific Git hash or DVC data version.

In [19]:
import hashlib
import json

def generate_data_fingerprint(df: pd.DataFrame, version_tag: str) -> dict:
    """
    Problem: Unchecked data changes breaking downstream models.
    Solution: Immutable data fingerprinting.
    """
    # Generate a hash of the current dataframe state
    data_hash = hashlib.sha256(pd.util.hash_pandas_object(df).values).hexdigest()

    manifest = {
        "version": version_tag,
        "hash": data_hash,
        "row_count": len(df),
        "columns": list(df.columns)
    }

    print(f"Data version {version_tag} locked with fingerprint: {data_hash[:10]}...")
    return manifest

---

## Module: ETL & Data Pipelines

**Introduction:** We solve the "Reliable Data Movement" problem by engineering scalable, fault-tolerant pipelines that transform raw assets into model-ready features.

### Chapter: ETL vs ELT Engineering Strategies

#### Strategic Suitability Framework (5 Factors)
1. **Problem Solved**: **Automating Data Transformation**. Replaces manual data prep with reproducible code execution.
2. **When to Use**: Production environments where data arrives continuously or in scheduled batches.
3. **When to Avoid**: Ad-hoc one-time analysis.
4. **Optimal Workflow**: **Extract → Validate → Transform → Load → Monitor**.
5. **Failure Modes**: **Dependency Hell**. Pipelines failing because an upstream table changed without notification.

**Interpreting Results:** A healthy pipeline is **Idempotent**—running it twice results in the same state without duplicating data.

## Module 1: Ingestion, Storage & Quality Foundations

**Introduction:** This module addresses the 'First Mile' of AI. We solve the problem of data reliability and memory constraints, ensuring the pipeline never consumes corrupt data or crashes on large files.

### Chapter 01: High-Throughput Ingestion & Storage Formats

#### Strategic Suitability Framework (5 Factors)
1. **Problem Solved**: **Memory Overflow & I/O Bottlenecks**. Prevents system crashes when reading multi-gigabyte files.
2. **When to Use**: Mandatory for 'Big Data' pipelines where dataset size > 20% of system RAM.
3. **When to Avoid**: Small configuration files or datasets under 500MB where CSV simplicity outweighs Parquet overhead.
4. **Optimal Workflow**: **Batch-Chunked Pipeline**. Read in small blocks, validate, and serialize immediately to disk.
5. **Failure Modes**: **Schema Evolution Mismatch**. Adding columns to a Parquet file that differ from the initial chunk's metadata.

**Interpreting Results:** When running the streaming function, success is measured by the lack of memory spikes. If your RAM usage stays flat while the Parquet file grows, your engineering pipeline is successfully decoupled from data scale.

In [ ]:
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

def stream_csv_to_parquet(csv_filepath: str, parquet_filepath: str, chunksize: int = 10_000) -> None:
    """Streams a large CSV file in chunks, enforces dynamic schema, and writes to Parquet."""
    first_chunk = True
    writer = None

    for chunk in pd.read_csv(csv_filepath, chunksize=chunksize):
        # Enforce basic numeric dynamic validation
        chunk['val'] = pd.to_numeric(chunk['val'], errors='coerce').fillna(0.0)
        table = pa.Table.from_pandas(chunk)

        if first_chunk:
            writer = pq.ParquetWriter(parquet_filepath, table.schema, compression='snappy')
            first_chunk = False

        writer.write_table(table)

    if writer:
        writer.close()

#### Failure Modes & Best Practices
* **Pitfall**: Appending directly to row-oriented structures in memory leading to Out-Of-Memory (OOM) errors.
* **Best Practice**: Stream large raw datasets using chunked iteration and store intermediate steps in columnar formats like Parquet.

---

### Chapter 02: Automated Data Quality Auditing

#### Strategic Suitability Framework (5 Factors)
1. **Problem Solved**: **Downstream Corruption**. Prevents 'garbage' data from polluting model training.
2. **When to Use**: Every production pipeline, especially those handling external user-generated content.
3. **When to Avoid**: Exploratory notebooks where quick iteration is preferred over strict adherence to bounds.
4. **Optimal Workflow**: **Assertive Gateway**. A blocking step in the pipeline that halts execution if quality drops below a threshold.
5. **Failure Modes**: **Over-Strict Filtering**. Silently dropping valid but 'extreme' edge cases, biasing the model against rare events.

**Interpreting Results:** The 'invalid rows dropped' printout serves as a health signal. A high drop rate suggests an upstream sensor failure or a breaking change in the data source schema.

In [ ]:
import pandas as pd
import numpy as np

def validate_dataframe(df: pd.DataFrame, schema_bounds: dict) -> pd.DataFrame:
    """Audits data quality and filters records failing basic bounded schema assertions."""
    valid_mask = pd.Series(True, index=df.index)

    for col, (min_val, max_val) in schema_bounds.items():
        if col in df.columns:
            column_mask = df[col].between(min_val, max_val) | df[col].isna()
            valid_mask &= column_mask

    cleaned_df = df[valid_mask].copy()
    print(f"Audited: {len(df) - len(cleaned_df)} invalid rows dropped.")
    return cleaned_df

#### Failure Modes & Best Practices
* **Pitfall**: Silently dropping invalid records without recording schema breach metrics.
* **Best Practice**: Assert schema conditions at the ingestion layer to prevent malformed data from propagating down the MLOps pipeline.

---

### Module 1: Business Case Studies

**Enterprise Case: Global Financial Transaction Ledger**
Large banks ingest billions of transactions daily. They use chunked Parquet streaming to ensure that fraud detection models can query 10 years of history without loading the entire dataset into RAM.

**Small Business Case: Local E-commerce Inventory Audit**
A small online shop uses dynamic schema validation to ensure that vendor-provided CSVs don't contain negative price values or missing SKU identifiers before updating the storefront.

In [6]:
import pandas as pd

# Small Business Inventory Audit Demo
def audit_inventory_update(csv_input: str):
    # Basic bounds for a small shop
    schema_rules = {'price': (0.01, 10000.0), 'stock_level': (0, 500)}
    df = pd.read_csv(csv_input)

    # Module 1 Principle: Immediate validation
    clean_df = validate_dataframe(df, schema_rules)
    return clean_df

## Module 2: Core Engineering, Cleaning & Feature Extraction

**Introduction:** This module focuses on signal recovery and stabilization. We solve for 'missingness' and 'scale variance,' ensuring the features provided to the model are statistically sound and comparable.

### Chapter 03: Missing Data Imputation

#### Strategic Suitability Framework (5 Factors)
1. **Problem Solved**: **Information Loss**. Recovers signal from incomplete records rather than discarding them.
2. **When to Use**: When data is Missing At Random (MAR) and the missingness correlates with other features.
3. **When to Avoid**: If missingness is over 50% for a feature, as the 'hallucinated' values will likely be noise.
4. **Optimal Workflow**: **Iterative Multivariate Imputation (MICE)**. Modeling missingness as a relationship between all variables.
5. **Failure Modes**: **Data Leakage**. Fitting an imputer on the full dataset before splitting into training and test sets.

**Interpreting Results:** Compare the variance of the feature before and after imputation. Success means the distribution remains similar without introducing artificial spikes at the mean.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

def impute_missing_multivariate(train_df: pd.DataFrame, test_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Fits an iterative multivariate imputer on train data and applies it to test data."""
    imputer = IterativeImputer(max_iter=10, random_state=42)

    train_imputed = imputer.fit_transform(train_train := train_df.select_dtypes(include=[np.number]))
    test_imputed = imputer.transform(test_df.select_dtypes(include=[np.number]))

    return (
        pd.DataFrame(train_imputed, columns=train_train.columns, index=train_df.index),
        pd.DataFrame(test_imputed, columns=train_train.columns, index=test_df.index)
    )

#### Failure Modes & Best Practices
* **Pitfall**: Imputing using global dataset metrics (mean/median) prior to train/test splits.
* **Best Practice**: Fit imputation models exclusively on training partitions, applying fitted parameters downstream.

---

### Chapter 04: Outlier Detection & Scaling

#### Strategic Suitability Framework (5 Factors)
1. **Problem Solved**: **Gradient Explosion/Vanishing**. Ensures feature scales don't dominate the model weight updates.
2. **When to Use**: Essential for Neural Networks and Distance-based models (KNN, SVM).
3. **When to Avoid**: Tree-based models (XGBoost, Random Forest) which are naturally invariant to feature scales.
4. **Optimal Workflow**: **Robust Quantile Scaling**. Using IQR to scale features without being pulled away by extreme outliers.
5. **Failure Modes**: **Ignoring Heavy Tails**. Normalizing a Power-Law distribution, which compresses the meaningful signal into a tiny range.

**Interpreting Results:** Visualizing a boxplot of the scaled features should show a distribution centered around 0 with most values between -2 and 2, regardless of original units.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import IsolationForest

def clean_and_scale_outliers(df: pd.DataFrame, feature_cols: list[str]) -> pd.DataFrame:
    """Detects multivariate outliers via Isolation Forest and scales robustly."""
    iso = IsolationForest(contamination=0.05, random_state=42)
    inlier_mask = iso.fit_predict(df[feature_cols]) != -1

    filtered_df = df[inlier_mask].copy()
    scaler = RobustScaler()
    filtered_df[feature_cols] = scaler.fit_transform(filtered_df[feature_cols])

    return filtered_df

#### Failure Modes & Best Practices
* **Pitfall**: Using standard z-score normalization on heavy-tailed distributions with untrimmed extreme outliers.
* **Best Practice**: Combine tree-based outlier isolation with quantile-based scaling.

---

### Chapter 05: Categorical Encoding & Dimensionality Reduction

#### Core Learning Objectives
* Contrast high-cardinality target encoding with sparse one-hot methods.
* Apply Principal Component Analysis (PCA) to reduce high-dimensional spaces while retaining target variance.
* Prevent target leakage when calculating target statistics.

#### Theoretical Concept
Target encoding replaces category $c$ with the target expectation $E[y|x=c]$. To prevent target leakage, smoothed target encoding uses a weighting factor $\lambda(n)$:
$$S_c = \lambda(n_c) \bar{y}_c + (1 - \lambda(n_c)) \bar{y}_{global}$$

In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA

def encode_and_reduce(df: pd.DataFrame, cat_col: str, target_col: str, num_cols: list[str], n_components: int = 2) -> pd.DataFrame:
    """Applies smoothed target encoding and PCA dimensionality reduction."""
    # Compute smoothed mean target
    global_mean = df[target_col].mean()
    stats = df.groupby(cat_col)[target_col].agg(['count', 'mean'])
    smooth = 10
    smoothed_vals = (stats['count'] * stats['mean'] + smooth * global_mean) / (stats['count'] + smooth)

    df[f"{cat_col}_encoded"] = df[cat_col].map(smoothed_vals).fillna(global_mean)

    # Dimensionality Reduction
    pca = PCA(n_components=n_components)
    pca_features = pca.fit_transform(df[num_cols])
    for i in range(n_components):
        df[f'pca_{i}'] = pca_features[:, i]

    return df

#### Failure Modes & Best Practices
* **Pitfall**: Applying one-hot encoding directly onto continuous high-cardinality string identifiers leading to dimensional explosion.
* **Best Practice**: Use target encoding or feature embeddings for high-cardinality features.

---

### Chapter 06: Class Imbalance & Resampling Strategies

#### Core Learning Objectives
* Resolve severe class distribution imbalances in target domains.
* Implement synthetic feature generation using SMOTE alongside selective undersampling.
* Properly align evaluation metrics to non-uniform label distributions.

#### Theoretical Concept
SMOTE (Synthetic Minority Over-sampling Technique) interpolates between minority class instances. For a sample $x_i$, a random $k$-nearest neighbor $x_{zi}$ is selected, and synthetic sample $x_{new}$ is created via:
$$x_{new} = x_i + \lambda (x_{zi} - x_i) \quad \text{where } \lambda \sim U(0,1)$$

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from imblearn.over_sampling import SMOTE

def balance_dataset(X: pd.DataFrame, y: pd.Series) -> tuple[pd.DataFrame, pd.Series]:
    """Applies SMOTE to rebalance class label distributions in training partitions."""
    smote = SMOTE(random_state=42)
    X_res, y_res = smote.fit_resample(X, y)
    return pd.DataFrame(X_res, columns=X.columns), pd.Series(y_res, name=y.name)

#### Failure Modes & Best Practices
* **Pitfall**: Oversampling the target dataset prior to applying train/validation splits, leading to shared synthetic samples across partitions.
* **Best Practice**: Apply synthetic oversampling exclusively to the training split post-split.

---

### Module 2: Business Case Studies

**Enterprise Case: Credit Scoring for Underbanked Populations**
Credit bureaus use Iterative Imputation (MICE) to fill in missing financial history for younger applicants, using existing features like utility payment consistency to predict missing credit card history accurately.

**Small Business Case: Customer Churn for a Subscription Box**
A small subscription service uses SMOTE to balance their dataset. Since most customers *don't* cancel, the 'Churn' class is tiny. SMOTE creates synthetic profiles of 'at-risk' customers to help the small marketing team target retention efforts.

In [7]:
# Enterprise Imputation Demo
def handle_enterprise_credit_gaps(client_data: pd.DataFrame):
    # Using Module 2's iterative imputer to maintain feature correlations
    train, test = client_data[:800], client_data[800:]
    imputed_train, _ = impute_missing_multivariate(train, test)
    return imputed_train

## Module 3: Modality Masterclasses

**Introduction:** We move beyond tabular data to engineer domain-specific modalities. This module solves the translation problem—converting unstructured human inputs (text, audio, vision) into mathematical tensors models can process.

### Chapter 07: Unstructured Text: Cleaning & Tokenization

#### Strategic Suitability Framework (5 Factors)
1. **Problem Solved**: **Vocab Explosion**. Breaks language into subwords to handle millions of words with a small vocabulary.
2. **When to Use**: All NLP tasks involving Transformers (BERT, GPT, Llama).
3. **When to Avoid**: Simple keyword-matching search engines where exact character matching is preferred.
4. **Optimal Workflow**: **Byte-Pair Encoding (BPE)**. Iteratively merging frequent character sequences.
5. **Failure Modes**: **Over-Normalization**. Removing emojis or punctuation that contain crucial sentiment or intent signal.

**Interpreting Results:** The output tensor shape `(Batch, Seq_Len)` indicates if your padding/truncation strategy is efficient. High padding percentage indicates a need for dynamic batching.

In [ ]:
import re
import torch
from transformers import AutoTokenizer

def prepare_text_batch(texts: list[str], model_name: str = "bert-base-uncased", max_length: int = 128) -> dict[str, torch.Tensor]:
    """Cleans text with regex and tokenizes into padded PyTorch tensors."""
    clean_texts = [re.sub(r"[^\w\s]", "", text.lower().strip()) for text in texts]

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    encoded = tokenizer(
        clean_texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )
    return encoded

#### Failure Modes & Best Practices
* **Pitfall**: Aggressively stripping symbols/punctuation needed for domain tasks like code generation or sentiment parsing.
* **Best Practice**: Align domain-specific regex cleaning rules with the pre-trained model's tokenization scheme.

---

### Chapter 08: Text Feature Extraction: Embeddings & Vector Stores

#### Core Learning Objectives
* Compute continuous vector representations for text blocks using pre-trained neural networks.
* Build chunking strategies optimized for downstream Retrieval-Augmented Generation (RAG).
* Index vectors for low-latency similarity queries.

#### Theoretical Concept
Cosine similarity evaluates the metric distance between dynamic sentence embeddings $u$ and $v$ within vector space $\mathbb{R}^d$:
$$\text{Sim}(u, v) = \frac{u \cdot v}{\|u\|_2 \|v\|_2}$$

In [ ]:
import numpy as np
import torch
from sklearn.metrics.pairwise import cosine_similarity

def create_embeddings_and_search(query: str, corpus: list[str], embed_fn) -> list[tuple[str, float]]:
    """Generates embeddings for corpus chunks and ranks them by cosine similarity."""
    corpus_embeddings = embed_fn(corpus)  # Expected shape: (N, D)
    query_embedding = embed_fn([query])   # Expected shape: (1, D)

    scores = cosine_similarity(query_embedding, corpus_embeddings)[0]
    ranked_indices = np.argsort(scores)[::-1]

    return [(corpus[idx], float(scores[idx])) for idx in ranked_indices]

#### Failure Modes & Best Practices
* **Pitfall**: Naively chunking documents purely by character count without taking structural boundaries (paragraphs, headings) into account.
* **Best Practice**: Use semantic or sentence-boundary chunking algorithms for vector store indexing.

---

### Chapter 09: Computer Vision: Preprocessing, Augmentation & Spatial Transforms

#### Core Learning Objectives
* Perform standardized spatial transforms, resizings, and color channel normalizations.
* Build online dataset augmentation pipelines to maximize variance during model training.
* Validate bounding boxes and mask transformations during visual geometry edits.

#### Theoretical Concept
Image normalization maps raw pixel intensities $P_{i,j,c} \in [0, 255]$ into standardized continuous values using channel-wise parameters:
$$\hat{P}_{i,j,c} = \frac{\frac{P_{i,j,c}}{255} - \mu_c}{\sigma_c}$$

In [ ]:
import torch
from torchvision import transforms
from PIL import Image

def get_vision_transform_pipeline(image_size: tuple[int, int] = (224, 224)) -> transforms.Compose:
    """Builds a image preprocessing and augmentation pipeline."""
    return transforms.Compose([
        transforms.Resize(image_size),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

#### Failure Modes & Best Practices
* **Pitfall**: Applying spatial transforms (e.g., cropping/flipping) to input images without adjusting corresponding object detection bounding boxes.
* **Best Practice**: Ensure visual augmentation libraries synchronized spatial transforms across image inputs and annotation layers.

---

### Chapter 10: Audio Preprocessing: Spectrograms & Feature Extraction

#### Core Learning Objectives
* Convert raw temporal audio signals into time-frequency representations.
* Compute Mel-Frequency Cepstral Coefficients (MFCCs) for speech processing workloads.
* Apply temporal trimming, padding, and gain normalizations to raw waveforms.

#### Theoretical Concept
The Short-Time Fourier Transform (STFT) computes discrete Fourier transforms over overlapping windowed segments of a continuous signal $x[n]$:
$$X(m, \omega) = \sum_{n=-\infty}^{\infty} x[n] w[n-m] e^{-j\omega n}$$
Mapping $|X(m, \omega)|^2$ to non-linear Mel filterbanks models human auditory perception.

In [ ]:
import numpy as np
import torch

def compute_mel_spectrogram(waveform: torch.Tensor, sample_rate: int = 16000, n_mels: int = 64) -> torch.Tensor:
    """Computes a normalized Mel-Spectrogram tensor from raw 1D audio waveforms."""
    # Ensure standard length
    max_len = sample_rate * 3 # 3 seconds
    if waveform.shape[-1] > max_len:
        waveform = waveform[..., :max_len]
    else:
        waveform = torch.nn.functional.pad(waveform, (0, max_len - waveform.shape[-1]))

    # Standard STFT signal transform placeholder representation via matrix operations
    stft = torch.stft(waveform, n_fft=400, hop_length=160, return_complex=True)
    spectrogram = torch.abs(stft)**2
    return spectrogram

#### Failure Modes & Best Practices
* **Pitfall**: Mixing audio inputs recorded at different sampling rates without standardizing resampling upfront.
* **Best Practice**: Convert all incoming audio assets to a single target sampling rate (e.g., 16kHz) early in ingestion.

---

### Chapter 11: Time Series: Feature Engineering & Windowing

#### Core Learning Objectives
* Convert continuous sequence streams into tabular sliding-window datasets.
* Engineer stationary time-series inputs using lag variables, rolling statistics, and differencing.
* Handle irregular temporal sampling and missing timestamp entries.

#### Theoretical Concept
To remove non-stationarity driven by trend/seasonality, first-order differencing is applied:
$$\Delta Y_t = Y_t - Y_{t-1}$$
Sliding window mappings construct target matrices $Y \in \mathbb{R}^{N \times h}$ from historical lags $X \in \mathbb{R}^{N \times w}$.

In [ ]:
import pandas as pd
import numpy as np

def create_sliding_windows(df: pd.DataFrame, target_col: str, window_size: int = 5, horizon: int = 1) -> tuple[np.ndarray, np.ndarray]:
    """Converts a univariate time series dataframe into sliding input/output arrays."""
    series = df[target_col].values
    X, y = [], []
    for i in range(len(series) - window_size - horizon + 1):
        X.append(series[i : i + window_size])
        y.append(series[i + window_size : i + window_size + horizon])
    return np.array(X), np.array(y)

#### Failure Modes & Best Practices
* **Pitfall**: Computing global rolling aggregates using future values (look-ahead bias / data leakage).
* **Best Practice**: Use strict trailing/backward windows when calculating moving averages or scaling temporal features.

---

### Chapter 12: Multimodal Processing & Multi-Tensor Fusion

#### Core Learning Objectives
* Align disparate modalities (e.g., text, images, tabular) along shared batch dimensions.
* Construct custom PyTorch `Dataset` structures designed for joint multi-tensor streaming.
* Apply cross-modal missing value masks during batch assembly.

#### Theoretical Concept
Multimodal fusion integrates distinct representation spaces $H_A \in \mathbb{R}^{d_A}$ and $H_B \in \mathbb{R}^{d_B}$. Late/Cross-Attention concatenation constructs a unified tensor space $H_{fused}$:
$$H_{fused} = [W_A H_A \,||\, W_B H_B] \quad \text{where } W_k \in \mathbb{R}^{d_{target} \times d_k}$$

In [ ]:
import torch
from torch.utils.data import Dataset

class MultimodalDataset(Dataset):
    """Custom dataset handling unified aligned indexing across text and image tensors."""
    def __init__(self, text_tensors: torch.Tensor, image_tensors: torch.Tensor, labels: torch.Tensor):
        assert len(text_tensors) == len(image_tensors) == len(labels)
        self.text = text_tensors
        self.images = image_tensors
        self.labels = labels

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        return {
            "text_inputs": self.text[idx],
            "image_inputs": self.images[idx],
            "label": self.labels[idx]
        }

#### Failure Modes & Best Practices
* **Pitfall**: Failing to handle unaligned indexing across modalities, leading to cross-contamination of targets and features.
* **Best Practice**: Enforce key verification across all input sources prior to constructing multi-tensor datasets.

---

### Module 3: Business Case Studies

**Enterprise Case: Multimodal Product Search (Visual & Text)**
Global retailers like Amazon use Multimodal Fusion. When you search for 'Blue Summer Dress', the system fuses your text query with image embeddings of inventory to find the exact visual match.

**Small Business Case: Podcast Transcription & Search**
A niche podcast creator uses Audio Spectrograms and Text Embeddings to allow listeners to search for specific moments in past episodes based on the 'vibe' (audio) or keywords (text).

In [8]:
# Small Business Podcast Search Demo
def index_podcast_segment(audio_clip, transcript_text):
    # Process audio via Module 3 MFCC logic
    audio_feat = compute_mel_spectrogram(audio_clip)
    # Process text via Module 3 Tokenization
    text_feat = prepare_text_batch([transcript_text])
    return {"audio": audio_feat, "text": text_feat}

## Module 4: Production, MLOps & Advanced AI Pipeline Engineering

**Introduction:** This module solves the 'Sustainability' problem. We address how to keep models accurate over time and how to protect user data according to global governance standards.

### Chapter 13: Feature Stores & Real-Time Optimization

#### Strategic Suitability Framework (5 Factors)
1. **Problem Solved**: **Train/Serve Skew**. Ensures features used in training are identical to those used in production.
2. **When to Use**: High-traffic apps requiring sub-10ms feature lookups (e.g., Recommendations, Fraud).
3. **When to Avoid**: Batch-only dashboards where real-time inference is not required.
4. **Optimal Workflow**: **Dual-Write Architecture**. Write features to an Offline Store (Parquet) and an Online Store (Redis) simultaneously.
5. **Failure Modes**: **Stale Features**. The online store failing to update, causing the model to predict based on week-old data.

**Interpreting Results:** Measure 'Latency vs. Staleness'. Success is characterized by sub-millisecond retrievals of features updated within the last few seconds.

In [ ]:
import time
import pandas as pd

class MockOnlineFeatureStore:
    """Simulates an online feature store reading from an in-memory key-value cache."""
    def __init__(self):
        self._store = {}

    def push_features(self, entity_id: str, feature_dict: dict) -> None:
        """Writes entity features to the online store."""
        self._store[entity_id] = {**feature_dict, "_timestamp": time.time()}

    def get_online_features(self, entity_ids: list[str]) -> list[dict]:
        """Fetches features for low-latency online inference."""
        return [self._store.get(eid, {}) for eid in entity_ids]

#### Failure Modes & Best Practices
* **Pitfall**: Re-implementing feature transformations separately in online vs. offline environments, leading to serving skew.
* **Best Practice**: Define transformation logic once in a central feature definition store.

---

### Chapter 14: Data Drift, Covariate Shift & Monitoring

#### Core Learning Objectives
* Measure distribution drift between reference training sets and incoming production inferences.
* Implement Statistical tests (KS-Test, Population Stability Index) to alert engineers to drift.
* Construct operational triggers for automated pipeline retraining.

#### Theoretical Concept
The **Population Stability Index (PSI)** quantifies variations between reference distribution $P$ and actual target distribution $Q$ across $k$ bins:
$$\text{PSI} = \sum_{b=1}^{k} \left( Q_b - P_b \right) \times \ln\left(\frac{Q_b}{P_b}\right)$$
$\text{PSI} > 0.2$ indicates significant distribution drift.

In [ ]:
import numpy as np
from scipy.stats import ks_2samp

def detect_covariate_shift(reference_data: np.ndarray, current_data: np.ndarray, alpha: float = 0.05) -> bool:
    """Uses a two-sample Kolmogorov-Smirnov test to detect feature distribution drift."""
    ks_stat, p_value = ks_2samp(reference_data, current_data)
    drift_detected = p_value < alpha
    print(f"KS Statistic: {ks_stat:.4f} | p-value: {p_value:.4f} | Drift Detected: {drift_detected}")
    return drift_detected

#### Failure Modes & Best Practices
* **Pitfall**: Monitoring model evaluation outputs exclusively while ignoring silent upstream feature drift.
* **Best Practice**: Run statistical validation checks on key incoming raw features continuously.

---

### Chapter 15: Privacy, Anonymization & Governance

#### Core Learning Objectives
* Anonymize Sensitive Personally Identifiable Information (PII) features using irreversible hashing/masking.
* Apply Differential Privacy (\epsilon, \delta) guarantees to statistical analytics outputs.
* Track end-to-end data lineage across processing jobs.

#### Theoretical Concept
A randomized mechanism $M$ satisfies $(\epsilon, \delta)$**-Differential Privacy** if for all neighboring datasets $D_1, D_2$ differing on a single record, and all query outputs $S \subseteq \text{Range}(M)$:
$$P[M(D_1) \in S] \le e^\epsilon \cdot P[M(D_2) \in S] + \delta$$

In [ ]:
import hashlib
import pandas as pd

def anonymize_pii_dataframe(df: pd.DataFrame, pii_cols: list[str], salt: str = "secure_salt") -> pd.DataFrame:
    """Hashes PII columns using SHA-256 with a secure salt."""
    anonymized_df = df.copy()
    for col in pii_cols:
        anonymized_df[col] = anonymized_df[col].astype(str).apply(
            lambda val: hashlib.sha256((val + salt).encode('utf-8')).hexdigest()
        )
    return anonymized_df

#### Failure Modes & Best Practices
* **Pitfall**: Hashing sensitive identifying attributes without a salt, leaving strings vulnerable to dictionary attack lookups.
* **Best Practice**: Combine cryptographic salting with strict role-based access control (RBAC) governance.

---

### Chapter 16: Synthetic Data Generation & Privacy-Preserving Augmentation

#### Core Learning Objectives
* Generate privacy-preserving synthetic tabular inputs using generative models or probabilistic distributions.
* Preserve cross-feature covariance structures without exposing original sensitive records.
* Evaluate statistical fidelity between real and generated distributions.

#### Theoretical Concept
Generative models sample from a learned continuous probability density function $p_\theta(x) \approx p_{data}(x)$. Sampling $\hat{x} \sim p_\theta(x)$ creates new instances that mimic the underlying distribution without duplicating original training records.

In [ ]:
import numpy as np
import pandas as pd

def generate_synthetic_gaussian_data(reference_df: pd.DataFrame, num_samples: int = 100) -> pd.DataFrame:
    """Generates synthetic tabular features using multivariate Gaussian sampling."""
    means = reference_df.mean()
    cov_matrix = reference_df.cov()

    synthetic_raw = np.random.multivariate_normal(mean=means, cov=cov_matrix, size=num_samples)
    return pd.DataFrame(synthetic_raw, columns=reference_df.columns)

#### Failure Modes & Best Practices
* **Pitfall**: Evaluating synthetic data solely on univariate distributions while ignoring distorted inter-feature correlations.
* **Best Practice**: Validate synthetic outputs using multivariate correlation matrices and downstream task evaluation scores.

### Module 4: Business Case Studies

**Enterprise Case: Real-time Credit Card Fraud Detection**
Credit card networks use Feature Stores (Online) to fetch a user's 'last 5 minutes of spending' in under 10ms to approve or deny a swipe at a grocery store.

**Small Business Case: Privacy-Preserving Loyalty Analytics**
A small cafe chain uses SHA-256 salting (Anonymization) to track customer return frequency without ever storing their actual names or emails on their local server, ensuring GDPR compliance.

In [9]:
# Small Business Privacy Demo
def secure_customer_loyalty(raw_emails: list):
    # Using Module 4 hashing to protect client identities
    df = pd.DataFrame({'email': raw_emails})
    return anonymize_pii_dataframe(df, ['email'], salt="Cafe_Secret_2024")

## Module 5: Relational Structures & Vectorized Knowledge

**Introduction:** This module solves for 'Contextual Awareness.' We engineer relational graphs and vector indices to allow models to understand the connections between entities, not just their individual values.

### Chapter 17: Graph-Based Data Engineering

#### Strategic Suitability Framework (5 Factors)
1. **Problem Solved**: **Relational Blindness**. Captures influence and connectivity that tabular data misses.
2. **When to Use**: Social networks, Supply Chain, Anti-Money Laundering (AML).
3. **When to Avoid**: Tabular data where entities have no logical connection or interactions.
4. **Optimal Workflow**: **Adjacency-List Engineering**. Representing entities as nodes and interactions as weighted edges.
5. **Failure Modes**: **Complexity Explosion**. Calculating global metrics like 'Betweenness' on million-node graphs.

**Interpreting Results:** High centrality scores (PageRank) indicate 'Influencer' nodes. In a fraud context, these are your high-risk entities.

In [ ]:
import networkx as nx
import pandas as pd
import numpy as np

def engineer_graph_features(edges_df: pd.DataFrame, source_col: str, target_col: str) -> pd.DataFrame:
    """Constructs a graph from edges and extracts topological centrality features."""
    G = nx.from_pandas_edgelist(edges_df, source=source_col, target=target_col)

    # Extract topological metrics
    pagerank = nx.pagerank(G)
    betweenness = nx.betweenness_centrality(G)
    clustering = nx.clustering(G)

    # Map features back to a node-level dataframe
    nodes = list(G.nodes())
    features = pd.DataFrame({
        'node_id': nodes,
        'pagerank_score': [pagerank[n] for n in nodes],
        'betweenness_score': [betweenness[n] for n in nodes],
        'clustering_coeff': [clustering[n] for n in nodes]
    })

    print(f"Extracted graph features for {len(nodes)} unique entities.")
    return features

#### Failure Modes & Best Practices
* **Pitfall**: Attempting to materialize global graph metrics on massive datasets (e.g., Betweenness Centrality) which has $O(V^3)$ complexity.
* **Best Practice**: Use sampling (random walks) or ego-graph analysis for large-scale production graph engineering.

---

### Chapter 18: Vector Database Engineering & HNSW Indexing

#### Core Learning Objectives
* Design ingestion pipelines for high-dimensional vector databases (Milvus, Pinecone, FAISS).
* Understand the trade-offs in Approximate Nearest Neighbor (ANN) indexing structures.
* Optimize metadata filtering strategies to reduce search space.

#### Theoretical Concept
**Hierarchical Navigable Small Worlds (HNSW)** is a state-of-the-art ANN algorithm. It builds a multi-layered graph where the top layers are coarse and the bottom layers are dense. Navigation occurs via a greedy search process:

$$\text{Objective: } \min_{v \in G} \text{dist}(q, v)$$

Where $q$ is the query vector and $v$ is a candidate in the proximity graph. This enables $O(\log(N))$ search latency on billion-scale datasets.

In [ ]:
import numpy as np

def prepare_vector_upsert_payload(ids: list, embeddings: np.ndarray, metadata: list[dict]) -> list[dict]:
    """
    Structures high-dimensional data for vector database ingestion.
    Ensures alignment between vectors and JSON metadata schema.
    """
    assert len(ids) == len(embeddings) == len(metadata)

    payload = []
    for i in range(len(ids)):
        record = {
            "id": str(ids[i]),
            "values": embeddings[i].tolist(), # Convert to list for API compatibility
            "metadata": metadata[i]
        }
        payload.append(record)

    print(f"Prepared payload with {len(payload)} vectors for upsert.")
    return payload

#### Failure Modes & Best Practices
* **Pitfall**: Storing raw high-resolution images/text inside the vector store metadata, leading to excessive IO and cost.
* **Best Practice**: Store only essential IDs and filtered metadata in the vector DB; store heavy binary blobs in object storage (S3/GCS).

---

### Chapter 19: Knowledge Graph Construction for RAG (GraphRAG)

#### Core Learning Objectives
* Implement entity-relation extraction pipelines to convert unstructured text into structured triplets.
* Engineer GraphRAG retrieval patterns that combine vector similarity with multi-hop graph traversal.
* Apply community detection algorithms to generate global summaries of segmented knowledge.

#### Theoretical Concept
Standard RAG often fails at global summarization. **GraphRAG** addresses this by partitioning the knowledge graph into hierarchical communities using the **Leiden Algorithm**.

For a query $Q$, retrieval involves finding the nearest entity nodes in vector space and then traversing their relational edges $E$ to collect context that is semantically distant but relationally connected:

$$\text{Context}(Q) = \{v \mid \text{dist}(\text{vec}(Q), \text{vec}(v)) < \tau\} \cup \text{Neighbors}_{G}(\{v\})$$

In [4]:
import networkx as nx

def extract_and_build_kg_triplets(extraction_results: list[dict]) -> nx.DiGraph:
    """
    Converts LLM-extracted entities and relations into a NetworkX directed graph.
    Expected extraction_results format: [{'head': 'AI', 'relation': 'enables', 'tail': 'Automation'}]
    """
    G = nx.DiGraph()

    for item in extraction_results:
        head = item['head']
        tail = item['tail']
        rel = item['relation']

        # Add nodes and edges with relational metadata
        G.add_edge(head, tail, relation=rel)

    print(f"Knowledge Graph built with {G.number_of_nodes()} entities and {G.number_of_edges()} relations.")
    return G

def get_multi_hop_context(G: nx.DiGraph, start_node: str, hops: int = 2) -> list[tuple]:
    """Extracts local neighborhood context for a specific entity up to N hops."""
    if not G.has_node(start_node):
        return []

    # Use Breadth-First Search to find local subgraph
    subgraph_nodes = nx.single_source_shortest_path_length(G, start_node, cutoff=hops).keys()
    subgraph = G.subgraph(subgraph_nodes)

    triplets = []
    for u, v, d in subgraph.edges(data=True):
        triplets.append((u, d['relation'], v))

    return triplets

In [5]:
# Sample triplet data extracted from hypothetical text
sample_triplets = [
    {'head': 'Large Language Model', 'relation': 'requires', 'tail': 'GPU Clusters'},
    {'head': 'GPU Clusters', 'relation': 'utilize', 'tail': 'H100 Nodes'},
    {'head': 'Large Language Model', 'relation': 'trained_on', 'tail': 'Massive Corpora'},
    {'head': 'Massive Corpora', 'relation': 'contains', 'tail': 'Unstructured Text'},
    {'head': 'GraphRAG', 'relation': 'enhances', 'tail': 'Large Language Model'},
    {'head': 'GraphRAG', 'relation': 'uses', 'tail': 'Knowledge Graphs'}
]

# 1. Build the Knowledge Graph
kg = extract_and_build_kg_triplets(sample_triplets)

# 2. Test Multi-hop retrieval for 'GraphRAG'
print("\nRetrieving 2-hop context for 'GraphRAG':")
context = get_multi_hop_context(kg, 'GraphRAG', hops=2)
for head, rel, tail in context:
    print(f"  - [{head}] --({rel})--> [{tail}]")

Knowledge Graph built with 7 entities and 6 relations.

Retrieving 2-hop context for 'GraphRAG':
  - [Large Language Model] --(requires)--> [GPU Clusters]
  - [Large Language Model] --(trained_on)--> [Massive Corpora]
  - [GraphRAG] --(enhances)--> [Large Language Model]
  - [GraphRAG] --(uses)--> [Knowledge Graphs]


#### Failure Modes & Best Practices
* **Pitfall**: Entity ambiguity (e.g., 'Apple' the company vs. 'Apple' the fruit) leading to corrupted graph paths.
* **Best Practice**: Use Entity Resolution (ER) or Entity Linking pipelines with a predefined taxonomy/ontology before graph insertion.
* **Pitfall**: Creating a 'dense' graph where every node connects to a central common node, leading to retrieval noise.
* **Best Practice**: Prune high-degree 'stop-nodes' (like generic terms) that don't add semantic value to specific queries.

### Module 5: Business Case Studies

**Enterprise Case: Supply Chain Resilience & Dependency Mapping**
Global manufacturing firms use Graph Engineering to map thousands of tiered suppliers. By calculating **Betweenness Centrality**, they identify 'bottleneck' suppliers whose failure would halt production across multiple continents.

**Small Business Case: Local Library Semantic Book Recommendations**
A local independent bookstore uses a Vector Database to power a recommendation engine. Customers find books with a similar 'mood' using high-dimensional embeddings of descriptions.

In [12]:
# Enterprise Supply Chain Demo
def identify_critical_suppliers(supplier_edges: pd.DataFrame):
    # Using Module 5 Graph logic to find systemic risks
    features = engineer_graph_features(supplier_edges, 'supplier_id', 'component_id')
    critical_nodes = features.sort_values('betweenness_score', ascending=False).head(5)
    return critical_nodes

### Module 5: Business Case Studies

**Enterprise Case: Supply Chain Resilience & Dependency Mapping**
Global manufacturing firms (e.g., Automotive) use Graph Engineering to map thousands of tiered suppliers. By calculating **Betweenness Centrality**, they identify 'bottleneck' suppliers whose failure would halt production across multiple continents.

**Small Business Case: Local Library Semantic Book Recommendations**
A local independent bookstore uses a Vector Database to power a recommendation engine. Instead of searching by keyword, customers can find books with a similar 'mood' or 'writing style' using high-dimensional embeddings of book descriptions.

In [10]:
# Enterprise Supply Chain Demo
def identify_critical_suppliers(supplier_edges: pd.DataFrame):
    # Using Module 5 Graph logic to find systemic risks
    features = engineer_graph_features(supplier_edges, 'supplier_id', 'component_id')
    critical_nodes = features.sort_values('betweenness_score', ascending=False).head(5)
    return critical_nodes

## Module 6: Privacy-Preserving Engineering

**Introduction:** We solve the 'Compliance and Safety' problem. This module engineers privacy directly into the data, allowing organizations to collaborate and train models without leaking individual secrets.

### Chapter 20: Engineering for Differential Privacy (DP)

#### Strategic Suitability Framework (5 Factors)
1. **Problem Solved**: **Re-identification Attacks**. Prevents bad actors from reverse-engineering individual data from aggregate reports.
2. **When to Use**: Sharing internal data with 3rd parties or public researchers.
3. **When to Avoid**: Critical precision tasks where adding noise makes the output useless.
4. **Optimal Workflow**: **Laplace Mechanism**. Calibrating noise to the sensitivity of the query and the privacy budget (Epsilon).
5. **Failure Modes**: **Budget Exhaustion**. Repeatedly querying until the 'privacy budget' is gone.

**Interpreting Results:** Success is achieving a 'Utility-Privacy Balance'. If the noisy mean is within 5% of the real mean but the Epsilon is low (e.g., < 1.0), your engineering is robust.

In [ ]:
import numpy as np

def apply_laplacian_noise(value: float, sensitivity: float, epsilon: float) -> float:
    """
    Adds calibrated Laplacian noise to a numerical aggregate to ensure epsilon-DP.
    """
    scale = sensitivity / epsilon
    noise = np.random.laplace(0, scale, 1)[0]
    return value + noise

# Example: Differentially Private Mean
def dp_mean(data: np.ndarray, epsilon: float, lower_bound: float, upper_bound: float) -> float:
    """
    Computes a DP mean. Sensitivity of sum is (upper - lower).
    """
    sensitivity = upper_bound - lower_bound
    noisy_sum = apply_laplacian_noise(np.sum(data), sensitivity, epsilon * 0.5)
    noisy_count = apply_laplacian_noise(len(data), 1, epsilon * 0.5)
    return noisy_sum / noisy_count

---

### Chapter 21: Federated Data Alignment & Schema Matching

#### Core Learning Objectives
* Align disparate schemas across decentralized 'Silos' without moving raw data.
* Implement Secure Multi-Party Computation (SMPC) placeholders for private joins.
* Engineer global feature maps from local site statistics.

#### Theoretical Concept
In **Federated Learning (FL)**, data preparation happens locally. The challenge is ensuring a **Global Schema** $\mathcal{S}_G$ that maps to local schemas $\mathcal{S}_L$:

$$\Phi: \mathcal{S}_L \to \mathcal{S}_G$$

This requires automated schema matching using semantic embeddings rather than exact string matches, as local headers often differ (e.g., `cust_id` vs `client_serial`).

In [ ]:
def federated_schema_check(local_columns: list[str], global_standard: list[str]) -> dict:
    """
    Verifies local site readiness for federated ingestion against a global standard.
    """
    missing = set(global_standard) - set(local_columns)
    extra = set(local_columns) - set(global_standard)

    status = {
        "is_ready": len(missing) == 0,
        "missing_features": list(missing),
        "unmapped_features": list(extra)
    }

    if not status["is_ready"]:
        print(f"[WARNING] Local site missing critical global features: {missing}")
    else:
        print("[SUCCESS] Local schema aligned with global federated standard.")

    return status

### Module 6: Business Case Studies

**Enterprise Case: Multi-Hospital Clinical Research (Federated)**
Hospitals use Federated Data Prep to ensure every site's local data (DICOM tags) matches a global standard before collaborative training begins, without sharing raw patient X-rays.

**Small Business Case: Privacy-Preserving Salary Surveys**
An HR consultancy uses Differential Privacy (Laplacian Noise) to release salary benchmarks, ensuring no individual's specific pay can be reverse-engineered from the report.

In [13]:
# Small Business Salary Survey Demo
def release_private_salary_avg(salary_data: list):
    # Using Module 6 DP Mean to protect individual privacy
    arr = np.array(salary_data)
    # Epsilon 0.1 (Strict privacy), Range 30k-200k
    return dp_mean(arr, epsilon=0.1, lower_bound=30000, upper_bound=200000)

### Module 6: Business Case Studies

**Enterprise Case: Multi-Hospital Clinical Research (Federated)**
A network of hospitals wants to train a pneumonia detection model without sharing private patient X-rays. They use Federated Data Prep to ensure every hospital's local data (DICOM tags) matches a global standard before training begins.

**Small Business Case: Privacy-Preserving Salary Surveys**
A local HR consultancy conducts salary benchmarks for small startups. They use Differential Privacy (Laplacian Noise) to release average salary data for roles like 'Junior Developer' ensuring that no individual's specific salary can be reverse-engineered from the report.

In [11]:
# Small Business Salary Survey Demo
def release_private_salary_avg(salary_data: list):
    # Using Module 6 DP Mean to protect individual privacy
    arr = np.array(salary_data)
    # Epsilon 0.1 (Strict privacy), Range 30k-200k
    return dp_mean(arr, epsilon=0.1, lower_bound=30000, upper_bound=200000)

---

## Module 7: Explainability, Fairness & Bias Engineering

### Chapter 22: Bias Detection & Mitigative Engineering

**Problem it solves:** Machine learning models often inherit and amplify historical human biases present in training data (e.g., gender or racial bias in hiring). This module engineers the data *before* training to ensure 'Individual Fairness' and 'Group Fairness'.

**When to use it:** Mandatory for high-stakes AI (Recruitment, Credit Scoring, Healthcare, Law Enforcement).
**When NOT to use it:** Low-stakes discovery tasks where the cost of data distortion exceeds the benefit of parity.

#### Theoretical Concept: Disparate Impact (DI)
We measure bias using the **Disparate Impact Ratio**. If the ratio of favorable outcomes for a protected group vs. a reference group is below 0.8 (the 4/5ths rule), the data is legally and ethically problematic.

**The Engineering Fix: Reweighing**
Instead of deleting data, we assign weights $W$ to instances to equalize the joint probability of the target $Y$ and the protected attribute $A$:
$$W = \frac{P(Y)P(A)}{P(Y, A)}$$

In [14]:
import pandas as pd
import numpy as np

def audit_and_reweigh(df, protected_attr, target_col):
    """
    Problem: Biased training labels.
    Solution: Calculate weights to achieve statistical parity.
    """
    total_n = len(df)
    # Probabilities of target
    p_y1 = df[target_col].mean()
    p_y0 = 1 - p_y1

    # Probabilities of protected attribute
    p_a1 = df[protected_attr].mean()
    p_a0 = 1 - p_a1

    weights = []
    for _, row in df.iterrows():
        y, a = row[target_col], row[protected_attr]
        # Observed joint probability
        p_ya = len(df[(df[target_col] == y) & (df[protected_attr] == a)]) / total_n
        # Target weight: (Independent P) / (Observed Joint P)
        expected_p = (p_y1 if y==1 else p_y0) * (p_a1 if a==1 else p_a0)
        weights.append(expected_p / p_ya)

    df['fairness_weight'] = weights
    return df

---

## Module 8: Data Curation & Engineering for Generative AI

### Chapter 23: Instruction Tuning & RLHF Data Prep

**Problem it solves:** Raw text data makes an LLM a 'document completer,' not an 'assistant.' To follow instructions, the data must be engineered into (Prompt, Response) pairs or (Prompt, Chosen, Rejected) triplets for Reinforcement Learning from Human Feedback (RLHF).

**When to use it:** When fine-tuning an LLM for a specific domain voice or task (e.g., medical advice, coding assistant).
**When NOT to use it:** If you are only using RAG (Retrieval) and do not intend to modify the model's weights.

#### Engineering Pattern: Preference Pairing
We curate datasets where humans rank responses. The goal is to maximize the margin between the 'Chosen' and 'Rejected' response tokens so the model learns the boundary of 'correctness'.

In [15]:
def curate_rlhf_triplets(interaction_logs: list):
    """
    Problem: Unstructured logs are unsuitable for RLHF.
    Solution: Pivot logs into Preference Pairs.
    """
    triplets = []
    for log in interaction_logs:
        prompt = log['query']
        # Identify best and worst based on human feedback score
        responses = sorted(log['outputs'], key=lambda x: x['score'], reverse=True)

        if len(responses) >= 2:
            triplets.append({
                "prompt": prompt,
                "chosen": responses[0]['text'],
                "rejected": responses[-1]['text']
            })

    return pd.DataFrame(triplets)

# Sample Data
logs = [{'query': 'Explain Python', 'outputs': [
    {'text': 'Python is a language.', 'score': 3},
    {'text': 'Python is an interpreted, high-level language...', 'score': 5}
]}]
display(curate_rlhf_triplets(logs))

,prompt,chosen,rejected
0,Explain Python,"Python is an interpreted, high-level language...",Python is a language.


### Modules 7 & 8: Business Context

**Enterprise Case: HR Bias Mitigation (Module 7)**
Large corporations engineering resume-screening pipelines must apply **Reweighing** (Chapter 22) to prevent historical gender imbalances from biasing the next generation of hires. Failure here leads to legal liability and poor talent acquisition.

**Small Business Case: Customer Support Chatbot (Module 8)**
A small agency converts 1,000 successful Slack support threads into **Instruction Pairs** (Chapter 23). This engineers a dataset that allows a small, cheap model (like Llama-3-8B) to sound exactly like their head of support without expensive prompt engineering.

---

## Module 9: Temporal Data Engineering & Forecasting Prep

### Chapter 24: Stationarity Engineering & Seasonal Decomposition

#### Strategic Suitability Framework (5 Factors)
1. **Problem Solved**: **Non-Stationarity**. Raw time-series often have shifting means (trends) and variances that confuse models into predicting based on 'recency bias' rather than 'patterns'.
2. **When to Use**: Mandatory for ARIMA, LSTMs, and XGBoost forecasting on financial, energy, or sensor data.
3. **When to Avoid**: Cross-sectional classification where the specific timestamp is just a metadata label and not a feature index.
4. **Optimal Workflow**: **Stateful Streaming Pipelines**. Requires maintaining a 'buffer' of previous events to calculate rolling windows or deltas in real-time.
5. **Failure Modes**: **Look-ahead Bias**. Calculating a rolling mean using future data during training, leading to 99% accuracy in the lab but total failure in production.

#### The Engineering Fix: Differencing
$$\Delta Y_t = Y_t - Y_{t-1}$$

In [16]:
import pandas as pd
import numpy as np

def engineer_stationary_series(df, col, lags=1):
    """
    Problem: Non-stationary trend causes model drift.
    Workflow: Sequential differencing pipeline.
    """
    # 1. Log transform to stabilize variance
    df[f'log_{col}'] = np.log1p(df[col])

    # 2. First-order differencing to stabilize mean
    df[f'diff_{col}'] = df[f'log_{col}'].diff(periods=lags)

    # 3. Rolling window stats (Temporal Feature Extraction)
    df[f'volatility_7d'] = df[f'diff_{col}'].rolling(window=7).std()

    return df.dropna()

---

## Module 10: Distributed Data Engineering for Big Data AI

### Chapter 25: Scaling Beyond a Single Machine (Pandas vs. Dask/Polars)

#### Strategic Suitability Framework (5 Factors)
1. **Problem Solved**: **The Memory Wall**. Solves Out-of-Memory (OOM) crashes when datasets exceed 20% of available RAM.
2. **When to Use**: Large-scale log analysis, clickstream processing, or billion-row feature stores.
3. **When to Avoid**: Datasets under 2GB. The overhead of parallel task management (scheduling/shuffling) makes distributed code 5x slower than simple Pandas on small data.
4. **Optimal Workflow**: **Lazy Evaluation & Directed Acyclic Graphs (DAGs)**. Build the 'recipe' of transformations first, then execute only when a result is needed.
5. **Failure Modes**: **Data Shuffling Bottlenecks**. Operations like 'Global Sort' or 'Global Join' force all data across the network, crashing the cluster cluster nodes via network congestion.

#### Theoretical Concept: Lazy Evaluation
Instead of `df.load()`, we use `df.scan()`. The engine optimizes the execution plan (Pushdown Filtering) before touching a single byte of data.

In [17]:
def explain_distributed_dag(total_data_size_gb):
    """
    Problem: 100GB Data on 16GB Machine.
    Solution: Chunked execution via Lazy Pipelines.
    """
    if total_data_size_gb > 10:
        workflow = "Distributed Lazy DAG (Dask/Spark/Polars)"
        strategy = "Partitioned Parallel Processing"
    else:
        workflow = "Eager In-Memory (Pandas)"
        strategy = "Vectorized CPU execution"

    print(f"Recommended Workflow: {workflow}")
    print(f"Core Strategy: {strategy}")
    return workflow

### Modules 9 & 10: Business Context

**Enterprise Case: Global Logistics Forecasting (Module 9)**
A shipping giant uses **Stateful Streaming** to predict port delays. By removing the seasonal 'Holiday Spike' from the data, their AI can detect genuine supply chain anomalies that would otherwise be hidden by predictable yearly cycles.

**Small Business Case: Scaling a Niche Search Engine (Module 10)**
A startup indexing 50 million niche web pages uses **Lazy Evaluation (Polars)**. This allowed them to run their feature engineering pipeline on a single cheap server rather than renting a massive, high-memory instance, saving 80% in cloud costs.

---

## Module 11: Explainability, Fairness & Bias Engineering

### Chapter 26: Bias Detection & Mitigative Engineering

#### Strategic Suitability Framework (5 Factors)
1. **Problem Solved**: **Historical Bias Amplification**. Models often inherit systemic biases (gender, race) from historical labels. This module engineers fairness directly into the training weights.
2. **When to Use**: Essential for 'High-Stakes' AI in Recruitment, Credit Scoring, Healthcare, and Law Enforcement.
3. **When to Avoid**: Low-stakes exploratory analysis where demographic parity is not a technical or ethical requirement.
4. **Optimal Workflow**: **Pre-processing Transformation**. Weights are calculated during the Batch Data Prep phase before model training begins.
5. **Failure Modes**: **Utility-Fairness Trade-off**. Over-correction can reduce overall model accuracy or predictive power for all groups.

#### The Engineering Fix: Reweighing
$$W = \frac{P(Y)P(A)}{P(Y, A)}$$

In [ ]:
import pandas as pd
import numpy as np

def calculate_fairness_weights(df, protected_attr, target_col):
    """
    Problem: Biased training labels for a specific group.
    Workflow: Pre-training weight adjustment.
    """
    # 1. Probabilities of target (Y) and protected attribute (A)
    p_y1 = df[target_col].mean()
    p_a1 = df[protected_attr].mean()
    p_y0 = 1 - p_y1
    p_a0 = 1 - p_a1

    # 2. Joint probability of privileged/unprivileged groups
    total = len(df)
    # Logic for Y=1, A=1 (Example weight calculation)
    p_y1a1 = len(df[(df[target_col] == 1) & (df[protected_attr] == 1)]) / total
    w_y1a1 = (p_y1 * p_a1) / p_y1a1 if p_y1a1 > 0 else 1.0

    print(f"Fairness weight for Y=1, A=1: {w_y1a1:.4f}")
    return w_y1a1

---

## Module 12: Data Curation & Engineering for Generative AI

### Chapter 27: RLHF & Instruction Tuning Data Pipelines

#### Strategic Suitability Framework (5 Factors)
1. **Problem Solved**: **Instruction Alignment**. Raw text data makes an LLM a 'completer'; this module engineers data to make it an 'assistant' that follows rules.
2. **When to Use**: Fine-tuning foundational models (Llama, Mistral) for a specific brand voice or safety alignment.
3. **When to Avoid**: Standard RAG where the base model already demonstrates sufficient reasoning and tone.
4. **Optimal Workflow**: **Preference Curation (Chosen vs. Rejected)**. Pitting multiple model responses against each other and ranking them.
5. **Failure Modes**: **Reward Hacking**. Curating data that rewards 'confident-sounding' responses regardless of factual accuracy.

#### The Engineering Fix: Preference Triplet Construction
Data is engineered into: `(Prompt, Chosen_Response, Rejected_Response)`

In [ ]:
def curate_preference_pairs(logs):
    """
    Problem: Unstructured chat logs for RLHF.
    Workflow: Ranking-based pivot pipeline.
    """
    triplets = []
    for log in logs:
        # Sort by human feedback score
        res = sorted(log['outputs'], key=lambda x: x['score'], reverse=True)
        if len(res) >= 2:
            triplets.append({
                "instruction": log['query'],
                "chosen": res[0]['text'],
                "rejected": res[-1]['text']
            })
    return pd.DataFrame(triplets)

# Sample usage logic
feedback_data = [{'query': 'Draft a legal NDA.', 'outputs': [
    {'text': 'Generic NDA text...', 'score': 3},
    {'text': 'Strict NDA with IP protection...', 'score': 5}
]}]
df_pairs = curate_preference_pairs(feedback_data)
display(df_pairs)

### Modules 11 & 12: Business Context

**Enterprise Case: Fairness in Automated Recruitment (Module 11)**
A global enterprise uses **Reweighing** to ensure their resume-screening AI doesn't unfairly penalize candidates from specific zip codes or gender groups, maintaining legal compliance and workforce diversity.

**Small Business Case: Brand-Specific Chatbot (Module 12)**
A small boutique agency curates **Instruction Pairs** from their best client communications. This allows them to fine-tune a cheap local LLM that mimics their specific helpful tone, saving thousands in monthly API costs for expensive base models.

# Task
The user wants to extend the 'Advanced AI Data Preparation & Pipeline Engineering' notebook with two new modules (SQL for AI Engineers and GPU-Accelerated Data Processing) and refine existing chapters. The task involves: 1) Adding Module 13 (SQL) with chapters on Window Functions, Recursive CTEs, and Data Contracts. 2) Adding Module 14 (GPU) focusing on RAPIDS/cuDF and zero-copy transfers. 3) Splitting the existing Imputation chapter in Module 2 into specialized KNN and MICE sections. 4) Adding specific Enterprise Case Studies for these new topics. 5) Finalizing the notebook by ensuring the 5-Factor Strategic Framework is applied consistently across all new content.

## Module 13: SQL for AI Engineers

### Subtask:
Add Module 13 focusing on advanced SQL techniques for feature engineering and data integrity.


## Module 13: SQL for AI Engineers

**Introduction:** We solve the 'Feature Leakage & Data Integrity' problem at the source. By pushing logic into the database layer, AI engineers ensure that feature engineering is consistent across training and production, utilizing the engine's optimized execution plans.

### Chapter 28: SQL Window Functions for Temporal Features

#### Strategic Suitability Framework (5 Factors)
1. **Problem Solved**: **Temporal Context Loss**. Captures trends, moving averages, and relative rankings within a dataset without complex self-joins.
2. **When to Use**: Preparing time-series features or user behavior sequences (e.g., 'last 5 actions').
3. **When to Avoid**: When the dataset is small enough for simple Pandas vectorization, as SQL overhead may not be justified.
4. **Optimal Workflow**: **Partitioning → Ordering → Framing**. Define the entity group, the sequence, and the look-back window.
5. **Failure Modes**: **Over-partitioning**. Creating windows so small that the statistical signal disappears into noise.

**Interpreting Results:** A successful window feature provides a column where every row contains context about its neighbors (e.g., a 7-day trailing average next to a daily sales figure).

**Reasoning**:
I will generate a code block using DuckDB to perform SQL window functions on a local DataFrame, creating temporal features like rolling averages and rankings.



In [20]:
import pandas as pd
import numpy as np
import duckdb

# 1. Create a dummy temporal dataset
data = {
    'date': pd.to_datetime(['2023-01-01', '2023-01-02', '2023-01-03', '2023-01-04', '2023-01-05'] * 2),
    'store_id': [1, 1, 1, 1, 1, 2, 2, 2, 2, 2],
    'sales': [100, 120, 110, 130, 150, 200, 210, 190, 220, 250]
}
sales_df = pd.DataFrame(data)

# 2. Use DuckDB to apply SQL Window Functions directly on the Pandas DF
query = """
SELECT
    *,
    AVG(sales) OVER (PARTITION BY store_id ORDER BY date ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) as rolling_avg_3d,
    RANK() OVER (PARTITION BY store_id ORDER BY sales DESC) as sales_rank_within_store
FROM sales_df
"""

feature_df = duckdb.query(query).to_df()

print("Features Engineered via SQL Window Functions:")
display(feature_df.head(10))

Features Engineered via SQL Window Functions:


,date,store_id,sales,rolling_avg_3d,sales_rank_within_store
0,2023-01-05,1,150,130.000000,1
1,2023-01-04,1,130,120.000000,2
2,2023-01-02,1,120,110.000000,3
3,2023-01-03,1,110,110.000000,4
4,2023-01-01,1,100,100.000000,5
5,2023-01-05,2,250,220.000000,1
6,2023-01-04,2,220,206.666667,2
7,2023-01-02,2,210,205.000000,3
8,2023-01-01,2,200,200.000000,4
9,2023-01-03,2,190,200.000000,5


### Chapter 29: Recursive CTEs for Graph & Hierarchy Features

#### Strategic Suitability Framework (5 Factors)
1. **Problem Solved**: **Relational Depth Bottlenecks**. Enables querying hierarchical data (org charts, BOMs) or performing path-finding without knowing the depth of the graph.
2. **When to Use**: Engineering features like 'Degrees of Separation' or 'Total Upstream Weight' in supply chain or social graphs.
3. **When to Avoid**: On cyclic graphs without termination logic, as it can trigger infinite loops.
4. **Optimal Workflow**: **Anchor Member → Union All → Recursive Member**. Establish the starting nodes, then iteratively join the table to itself.
5. **Failure Modes**: **Infinite Recursion**. Forgetting to include a depth limit or a cycle-detection flag.

**Interpreting Results:** Success is achieved when a flat adjacency list is transformed into a reachable path or a depth-level feature for every node.

**Reasoning**:
I will use DuckDB to execute a recursive SQL query on a pandas DataFrame to calculate the depth of employees in a company hierarchy.



In [21]:
import pandas as pd
import duckdb

# 1. Define a hierarchical adjacency list (Manager-Employee)
hierarchy_data = {
    'emp_id': [1, 2, 3, 4, 5, 6],
    'name': ['CEO', 'VP_Sales', 'VP_Eng', 'Sales_Mgr', 'Eng_Mgr', 'Junior_Dev'],
    'manager_id': [None, 1, 1, 2, 3, 5]
}
hierarchy_df = pd.DataFrame(hierarchy_data)

# 2. Recursive CTE to calculate hierarchy depth
recursive_query = """
WITH RECURSIVE org_path AS (
    -- Anchor Member: Start with the top-level node (CEO)
    SELECT emp_id, name, manager_id, 0 AS level
    FROM hierarchy_df
    WHERE manager_id IS NULL

    UNION ALL

    -- Recursive Member: Join the CTE back to the base table
    SELECT e.emp_id, e.name, e.manager_id, op.level + 1
    FROM hierarchy_df e
    JOIN org_path op ON e.manager_id = op.emp_id
    WHERE op.level < 10 -- Safety termination depth
)
SELECT * FROM org_path ORDER BY level ASC;
"""

hierarchy_features = duckdb.query(recursive_query).to_df()

print("Hierarchical Features Engineered via Recursive CTE:")
display(hierarchy_features)

Hierarchical Features Engineered via Recursive CTE:


,emp_id,name,manager_id,level
0,1,CEO,NaN,0
1,2,VP_Sales,1.0,1
2,3,VP_Eng,1.0,1
3,4,Sales_Mgr,2.0,2
4,5,Eng_Mgr,3.0,2
5,6,Junior_Dev,5.0,3


### Chapter 30: Data Contracts & Schema Enforcement

#### Strategic Suitability Framework (5 Factors)
1. **Problem Solved**: **Downstream Pipeline Breaks**. Prevents malformed data from entering the warehouse and breaking production models.
2. **When to Use**: In multi-team environments where data producers and consumers are decoupled.
3. **When to Avoid**: Early-stage exploratory research where schema agility is more important than strict enforcement.
4. **Optimal Workflow**: **Define Specification → Validation Gateway → Enforced Ingestion**. Use YAML or JSON schemas to validate incoming batches.
5. **Failure Modes**: **Schema Rigidity**. Overly strict contracts that prevent valid business evolutions from being captured.

**Interpreting Results:** A successful contract implementation results in 'Safe' ingestion logs where every record is guaranteed to match the expected feature types and ranges.

**Reasoning**:
Generate a code block implementing a data contract validation gateway using Pydantic to enforce data types and value ranges.



In [23]:
from pydantic import BaseModel, Field, ValidationError
from typing import List, Optional
import pandas as pd

# 1. Define the Data Contract (Schema)
class FeatureContract(BaseModel):
    item_id: int
    price: float = Field(gt=0, description='Price must be positive')
    category: str
    stock_status: Optional[str] = 'in_stock'

def validate_ingestion_batch(data: List[dict]):
    """
    Acts as a Gateway: Enforces the contract before data touches the DB.
    """
    valid_records = []
    errors = []

    for i, record in enumerate(data):
        try:
            validated = FeatureContract(**record)
            # Use model_dump() instead of the deprecated .dict()
            valid_records.append(validated.model_dump())
        except ValidationError as e:
            errors.append(f"Row {i} failed: {e.json()}")

    print(f"Successfully validated {len(valid_records)} records.")
    if errors:
        print(f"Rejected {len(errors)} records due to contract breach.")

    return pd.DataFrame(valid_records)

# 2. Simulate incoming 'dirty' data
raw_data = [
    {'item_id': 101, 'price': 25.50, 'category': 'A'},
    {'item_id': 102, 'price': -5.00, 'category': 'B'}, # Breach: negative price
    {'item_id': 103, 'price': 10.00, 'category': 'A', 'stock_status': 'out_of_stock'}
]

contracted_df = validate_ingestion_batch(raw_data)
display(contracted_df)

Successfully validated 2 records.
Rejected 1 records due to contract breach.


,item_id,price,category,stock_status
0,101,25.5,A,in_stock
1,103,10.0,A,out_of_stock


## Module 14: GPU-Accelerated Data Processing

### Subtask:
Add Module 14 focusing on GPU-bound engineering patterns using RAPIDS (cuDF) and zero-copy memory transfers.


## Module 14: GPU-Accelerated Data Processing

**Introduction:** We solve the 'Throughput & Latency' problem for massive datasets. As data scales into the hundreds of gigabytes, CPU-bound processing becomes a bottleneck. By offloading feature engineering to the GPU, we achieve 10x-100x speedups, enabling near-real-time iteration on large-scale models.

### Chapter 31: RAPIDS & cuDF for Accelerated Tabular Prep

#### Strategic Suitability Framework (5 Factors)
1. **Problem Solved**: **I/O and Compute Bottlenecks**. Accelerates operations like joins, group-bys, and string processing that are slow on single-core CPUs.
2. **When to Use**: When working with datasets that fit in GPU memory (e.g., 16GB-80GB) and require intensive feature extraction.
3. **When to Avoid**: Small datasets (< 1GB) where the overhead of moving data from Host (RAM) to Device (VRAM) exceeds the computation time.
4. **Optimal Workflow**: **Ingest (Parquet) → GPU Transform → Zero-Copy Handoff to DL Framework**.
5. **Failure Modes**: **Out-Of-Memory (OOM) on Device**. GPUs have less memory than system RAM; failing to manage memory pools leads to hard crashes.

**Interpreting Results:** A successful GPU pipeline shows high 'Device Utilization' and a significant reduction in execution time compared to Pandas benchmarks.

**Reasoning**:
I will provide a code block that simulates a large-scale data processing task using cuDF. Note: This code assumes a GPU-enabled environment. I will use a fallback or simulation if the library is missing to maintain the instructional value.



In [24]:
try:
    import cudf
    import numpy as np

    # 1. Generate a synthetic large dataset on the GPU
    n_rows = 1_000_000
    gdf = cudf.DataFrame({
        'user_id': np.random.randint(0, 1000, n_rows),
        'transaction_val': np.random.rand(n_rows) * 1000
    })

    # 2. Accelerated Feature Engineering: Group-by aggregation on Device
    # This executes in parallel across thousands of CUDA cores
    gpu_features = gdf.groupby('user_id').agg({
        'transaction_val': ['mean', 'max', 'std']
    })

    print("GPU-Accelerated Aggregation Results (cuDF):")
    print(gpu_features.head())

except ImportError:
    print("[INFO] RAPIDS cuDF not found in this environment.")
    print("In a GPU-enabled Colab instance, you would use: import cudf")
    print("cuDF mirrors the Pandas API but executes operations on the GPU (VRAM).")

[INFO] RAPIDS cuDF not found in this environment.
In a GPU-enabled Colab instance, you would use: import cudf
cuDF mirrors the Pandas API but executes operations on the GPU (VRAM).


### Chapter 32: DL-Pack & Zero-Copy Handoffs

#### Strategic Suitability Framework (5 Factors)
1. **Problem Solved**: **Redundant Serialization**. Eliminates the need to copy data from GPU memory to CPU RAM and back to GPU when moving between cuDF and PyTorch/TensorFlow.
2. **When to Use**: When transitioning from feature engineering (cuDF) to model training (PyTorch) within the same GPU memory space.
3. **When to Avoid**: When the training framework and preprocessing framework are on different physical machines or GPUs.
4. **Optimal Workflow**: **cuDF Transform → `to_dlpack()` → `from_dlpack()` → Model Training**.
5. **Failure Modes**: **Pointer Invalidity**. Accessing a tensor after the underlying cuDF buffer has been freed.

### Chapter 33: GPU Memory Pool Management (RMM)

#### Strategic Suitability Framework (5 Factors)
1. **Problem Solved**: **Allocation Latency**. Avoids the overhead of repeatedly asking the OS for GPU memory by pre-allocating a large 'Pool'.
2. **When to Use**: High-frequency processing tasks or when running multiple concurrent GPU jobs.
3. **When to Avoid**: Simple one-off scripts where memory allocation time is negligible.
4. **Optimal Workflow**: **Initialize RMM Pool → Execute Pipeline → Monitor Throughput**.
5. **Failure Modes**: **Pool Fragmentation**. Over-allocating many small buffers that prevent large contiguous allocations even if total free memory is high.

**Interpreting Results:** A healthy RMM configuration shows stable VRAM usage without the 'sawtooth' pattern of frequent allocations and deallocations.

**Reasoning**:
I am providing the executable code for Chapters 32 and 33 to demonstrate how to perform zero-copy handoffs and manage GPU memory pools efficiently.



In [25]:
try:
    import cudf
    import rmm
    import torch
    from torch.utils.dlpack import from_dlpack

    # 1. Chapter 33: Initialize RMM Pool
    # Pre-allocates 2GB of VRAM to prevent allocation overhead
    rmm.reinitialize(
        pool_allocator=True,
        initial_pool_size=2 * 1024**3,
        maximum_pool_size=4 * 1024**3
    )
    print("RMM Pool Initialized: 2GB pre-allocated.")

    # 2. Generate data on GPU via cuDF
    gdf = cudf.DataFrame({'feature': [1.0, 2.0, 3.5, 4.2]})

    # 3. Chapter 32: Zero-Copy Handoff via DLPack
    # Converts cuDF buffer to PyTorch tensor without touching CPU RAM
    dl_tensor = gdf.to_dlpack()
    th_tensor = from_dlpack(dl_tensor)

    print("Zero-Copy Handoff Success.")
    print(f"PyTorch Tensor on Device: {th_tensor.device}")
    print(th_tensor)

except ImportError:
    print("[INFO] GPU Engineering libraries (rmm/cudf) not found.")
    print("Pattern for Chapter 32 (Zero-Copy): tensor = torch.from_dlpack(gdf.to_dlpack())")
    print("Pattern for Chapter 33 (RMM): rmm.reinitialize(pool_allocator=True)")

[INFO] GPU Engineering libraries (rmm/cudf) not found.
Pattern for Chapter 32 (Zero-Copy): tensor = torch.from_dlpack(gdf.to_dlpack())
Pattern for Chapter 33 (RMM): rmm.reinitialize(pool_allocator=True)


## Chapter Splitting: Advanced Statistical Imputation

### Subtask:
Split the existing Imputation chapter in Module 2 into specialized KNN and MICE sections.


### Chapter 03A: K-Nearest Neighbors (KNN) Imputation

#### Strategic Suitability Framework (5 Factors)
1. **Problem Solved**: **Local Similarity Recovery**. Uses the 'k' most similar records to estimate missing values based on feature space proximity.
2. **When to Use**: Small to medium datasets where local clusters of similar observations are expected to have similar feature values.
3. **When to Avoid**: **High-Dimensional Data**. As dimensions increase, the distance between points becomes uniform (Curse of Dimensionality), making 'neighbors' less meaningful.
4. **Optimal Workflow**: **Scale Features → Select K → Compute Distances → Impute**.
5. **Failure Modes**: **Outlier Sensitivity**. Extreme values in the neighborhood can disproportionately bias the imputed value.

**Interpreting Results:** Success is measured by maintaining local density structures. Check if the imputed values fall within the expected range of their nearest neighbor clusters.

---

### Chapter 03B: Iterative Multivariate Imputation (MICE)

#### Strategic Suitability Framework (5 Factors)
1. **Problem Solved**: **Feature-to-Feature Correlation**. Models each feature with missing values as a function of other features in a round-robin fashion.
2. **When to Use**: When features have strong linear or non-linear relationships and data is Missing at Random (MAR).
3. **When to Avoid**: **Missing Not At Random (MNAR)** data where the reason for missingness isn't captured by other features, or very sparse datasets where models fail to converge.
4. **Optimal Workflow**: **Initialize (Mean/Median) → Regress Each Feature → Iterate to Convergence**.
5. **Failure Modes**: **Multicollinearity**. Highly redundant features can make the internal regression models unstable.

**Interpreting Results:** Success is indicated by stable convergence of the feature distributions over multiple iterations, preserving the global covariance matrix.

**Reasoning**:
I will provide the specialized code implementation for both KNN and MICE imputation, demonstrating how to use scikit-learn's KNNImputer and IterativeImputer within an engineering pipeline.



**Reasoning**:
Fix the ImportError by explicitly importing `enable_iterative_imputer` from `sklearn.experimental` as required by the library's API for this experimental feature.



In [27]:
import numpy as np
import pandas as pd
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import KNNImputer, IterativeImputer
from sklearn.preprocessing import StandardScaler

def impute_knn(df: pd.DataFrame, n_neighbors: int = 5) -> pd.DataFrame:
    """
    Implementation for Chapter 03A: KNN Imputation.
    Note: KNN is distance-based, so features must be scaled first.
    """
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df)

    imputer = KNNImputer(n_neighbors=n_neighbors)
    imputed_data = imputer.fit_transform(scaled_data)

    # Inverse transform to return to original scale
    final_data = scaler.inverse_transform(imputed_data)
    return pd.DataFrame(final_data, columns=df.columns, index=df.index)

def impute_mice(df: pd.DataFrame, max_iter: int = 10) -> pd.DataFrame:
    """
    Implementation for Chapter 03B: MICE (Iterative) Imputation.
    Models each feature as a function of others.
    """
    imputer = IterativeImputer(max_iter=max_iter, random_state=42)
    imputed_data = imputer.fit_transform(df)
    return pd.DataFrame(imputed_data, columns=df.columns, index=df.index)

# Example Usage
sample_data = pd.DataFrame({
    'feature_a': [1.2, 1.5, np.nan, 1.9, 2.1],
    'feature_b': [10, np.nan, 12, 14, 15]
})

print("KNN Imputed Data:")
display(impute_knn(sample_data))

print("\nMICE Imputed Data:")
display(impute_mice(sample_data))

KNN Imputed Data:


,feature_a,feature_b
0,1.200000,10.0
1,1.500000,13.0
2,1.733333,12.0
3,1.900000,14.0
4,2.100000,15.0



MICE Imputed Data:


,feature_a,feature_b
0,1.200000,10.000000
1,1.500000,11.694723
2,1.554364,12.000000
3,1.900000,14.000000
4,2.100000,15.000000


## Enterprise Case Studies: SQL & GPU Workloads

### Subtask:
Add specific business case studies and demo functions for Module 13 (SQL) and Module 14 (GPU).


### Modules 13 & 14: Business Case Studies

**Enterprise Case: Fintech Real-time Fraud Pattern Detection (Module 13)**
Digital banks use **SQL Window Functions** (Chapter 28) to analyze transaction streams. By partitioning by `card_id` and ordering by time, they calculate the 'velocity' of spending (e.g., more than 3 transactions in different cities within 1 hour). This allows the ingestion layer to flag suspicious activity before the data even reaches the model training cluster.

**Enterprise Case: Biotech Genomic Sequence Alignment (Module 14)**
Genomics companies process petabytes of short-read DNA sequences. Using **GPU-Accelerated Dataframes (cuDF)**, they perform massive parallel joins between patient samples and reference genomes. What used to take days on a CPU cluster now completes in hours, significantly accelerating the pipeline for personalized medicine discovery.

**Reasoning**:
I will create a code block containing two demo functions: one using SQL-like window logic for fintech fraud detection and another simulating a high-throughput GPU join for biotech research.



In [28]:
import pandas as pd
import duckdb

# 1. Fintech Fraud Detection Demo (Module 13 Logic)
def detect_transaction_velocity(tx_df: pd.DataFrame):
    """
    Uses SQL Window functions to detect 'velocity' of spending.
    """
    query = """
    SELECT
        *,
        COUNT(*) OVER (PARTITION BY card_id ORDER BY timestamp RANGE BETWEEN INTERVAL 1 HOUR PRECEDING AND CURRENT ROW) as tx_count_1h,
        SUM(amount) OVER (PARTITION BY card_id ORDER BY timestamp RANGE BETWEEN INTERVAL 1 HOUR PRECEDING AND CURRENT ROW) as tx_sum_1h
    FROM tx_df
    """
    return duckdb.query(query).to_df()

# 2. Biotech Genomic Processing Simulation (Module 14 Logic)
def gpu_simulated_sequence_join(patient_samples, reference_genome):
    """
    Simulates a high-throughput join typically performed on GPU (cuDF).
    In production, this would use `cudf.merge()`.
    """
    print("Simulation: Performing parallel GPU join on sequence markers...")
    # Fallback to pandas for simulation
    aligned_data = patient_samples.merge(reference_genome, on='marker_id', how='inner')
    return aligned_data

# Example usage logic
tx_sample = pd.DataFrame({
    'card_id': [101, 101, 101, 102],
    'timestamp': pd.to_datetime(['2024-01-01 10:00:00', '2024-01-01 10:20:00', '2024-01-01 10:45:00', '2024-01-01 11:00:00']),
    'amount': [50, 150, 200, 30]
})

velocity_df = detect_transaction_velocity(tx_sample)
print("Fintech Velocity Analysis:")
display(velocity_df)

Fintech Velocity Analysis:


,card_id,timestamp,amount,tx_count_1h,tx_sum_1h
0,102,2024-01-01 11:00:00,30,1,30.0
1,101,2024-01-01 10:00:00,50,1,50.0
2,101,2024-01-01 10:20:00,150,2,200.0
3,101,2024-01-01 10:45:00,200,3,400.0


## Final Task

### Subtask:
Finalize the notebook by summarizing the newly added modules and verifying consistency.


## Final Summary & Roadmap

### Extension Overview
We have successfully extended the **Advanced AI Data Preparation & Pipeline Engineering** curriculum with two critical advanced modules and refined statistical imputation techniques:

1.  **Module 13: SQL for AI Engineers**: Focused on moving feature engineering closer to the data source. We explored **Window Functions** for temporal features, **Recursive CTEs** for hierarchical data, and **Data Contracts** to enforce quality at the ingestion gateway.
2.  **Module 14: GPU-Accelerated Data Processing**: Addressed the 'Memory Wall' by utilizing **RAPIDS cuDF** for high-throughput tabular preparation and **DLPack** for zero-copy handoffs to Deep Learning frameworks like PyTorch.
3.  **Refined Imputation**: Re-engineered Chapter 03 into specialized sections for **KNN** and **MICE**, providing clear guidance on when to prioritize local similarity versus global feature correlations.

### The 5-Factor Strategic Framework
Every chapter in this notebook adheres to the 5-Factor Framework to ensure engineering decisions are grounded in tactical reality:
*   **Problem Solved**: The specific technical bottleneck addressed.
*   **When to Use**: The optimal deployment scenario.
*   **When to Avoid**: Counter-indicators and edge cases.
*   **Optimal Workflow**: The step-by-step engineering pattern.
*   **Failure Modes**: Common pitfalls and how to monitor for them.

---

**Conclusion:** Data engineering is not just about cleaning; it is about building resilient, performant, and fair pipelines. By mastering these modules—from SQL-bound logic to GPU-accelerated tensors—engineers can build AI systems that are truly production-ready.

# Task
Extend the 'Advanced AI Data Preparation & Pipeline Engineering' notebook by adding Module 13 (SQL for AI Engineers) covering window functions, recursive CTEs, and data contracts, and Module 14 (GPU-Accelerated Data Processing) focusing on RAPIDS/cuDF and zero-copy transfers. Additionally, split the existing Imputation chapter in Module 2 into specialized KNN and MICE sections, add corresponding enterprise case studies, and ensure the 5-Factor Strategic Framework is applied consistently throughout the new content.

## Summary:

### Q&A
- **How can SQL be used for AI engineering?** By utilizing Window Functions for temporal features and Recursive CTEs for hierarchical data, engineers can push feature logic to the database layer for consistency and performance. Data Contracts (via Pydantic) ensure schema integrity at the ingestion gateway.
- **How does GPU acceleration help in data processing?** RAPIDS cuDF provides a Pandas-like API that executes on CUDA cores, significantly increasing throughput for large datasets. Zero-copy transfers (DLPack) allow seamless handoffs between cuDF and PyTorch without CPU overhead.
- **What is the difference between KNN and MICE imputation?** KNN recovers local similarity by using distance-based neighbors (requiring feature scaling), whereas MICE (Multivariate Imputation by Chained Equations) models feature-to-feature correlations globally through iterative regression.

### Data Analysis Key Findings
- **SQL Window Functions**: Successfully calculated 3-day rolling averages and within-store sales rankings using DuckDB directly on Pandas DataFrames.
- **Recursive CTEs**: Transformed a flat employee-manager adjacency list into a hierarchical depth map.
- **Data Contracts**: Implemented a validation gateway that successfully filtered out records with negative prices (contract breaches) while preserving valid entries.
- **Imputation**: Demonstrated specialized KNN and MICE implementations, ensuring the experimental `IterativeImputer` was correctly initialized.
- **Fraud Detection**: Engineered a transaction velocity feature using SQL-based time-range windows to count events within a trailing 1-hour window.

### Insights or Next Steps
- **Standardize Ingestion**: Use the Data Contract pattern across all production pipelines to prevent 'silent' failures caused by schema drift from upstream sources.
- **GPU Handoffs**: For large-scale deep learning, implement the zero-copy DLPack pattern to avoid the costly I/O bottleneck of moving tensors between system RAM and VRAM.
